# MSc Dissertation — Progress Demo (Part 1 & 2)

**Author:** Sara Belhaiba  
**Supervisor:** Dr Christian Wagner  
**Module:** COMP4120 — MSc Cyber-Physical Systems (Cyber Security Pathway)  
University of Nottingham, 2025-2026

This notebook covers:
1. Data Preparation
2. Model Training (Random Forest & XGBoost)


## 1. Data Preparation

In [ ]:
"""
DISSERTATION CODE - PART 1: DATA PREPARATION
=============================================
This is the first part of the fraud detection system.
Run this before Part 2.

Author: Sara Belhaiba
Supervisor: Dr Christian Wagner
University of Nottingham - MSc Cyber-Physical Systems
"""

# =============================================================================
# STEP 1: Import Libraries
# =============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("PART 1: DATA PREPARATION")
print("=" * 60)

# =============================================================================
# STEP 2: Load Dataset
# =============================================================================
print("\n[1/4] Loading dataset...")

# Load the CSV file
df = pd.read_csv('data/creditcard.csv')

print(f"✓ Dataset loaded: {len(df):,} transactions")
print(f"✓ Columns: {len(df.columns)}")

# =============================================================================
# STEP 3: Explore Data
# =============================================================================
print("\n[2/4] Exploring data...")

# Count frauds
total = len(df)
frauds = df['Class'].sum()
legitimate = total - frauds
fraud_pct = (frauds / total) * 100

print(f"✓ Total transactions: {total:,}")
print(f"✓ Legitimate: {legitimate:,} ({100 - fraud_pct:.3f}%)")
print(f"✓ Fraudulent: {frauds:,} ({fraud_pct:.3f}%)")
print(f"✓ Class imbalance ratio: 1:{int(legitimate/frauds)}")

# =============================================================================
# STEP 4: Preprocess Data
# =============================================================================
print("\n[3/4] Preprocessing data...")

# Separate features (X) and target (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Scale Amount and Time
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])
X['Time'] = scaler.fit_transform(X[['Time']])

print("✓ Amount and Time normalised (StandardScaler)")

# =============================================================================
# STEP 5: Split Data
# =============================================================================
print("\n[4/4] Splitting data...")

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"✓ Training set: {len(X_train):,} transactions ({y_train.sum()} frauds)")
print(f"✓ Test set: {len(X_test):,} transactions ({y_test.sum()} frauds)")

# =============================================================================
# SAVE FOR PART 2
# =============================================================================
print("\n" + "=" * 60)
print("PART 1 COMPLETE")
print("=" * 60)
print("\nVariables ready for Part 2:")
print("  - X_train, X_test (features)")
print("  - y_train, y_test (labels)")
print("\nNext: Run Part 2 to train the models.")


## 2. Model Training — Random Forest & XGBoost

In [ ]:
"""
DISSERTATION CODE - PART 2: MODEL TRAINING
==========================================
This part trains the two models: Random Forest and XGBoost.
Run Part 1 first!

Author: Sara Belhaiba
Supervisor: Dr Christian Wagner
University of Nottingham - MSc Cyber-Physical Systems
"""

# =============================================================================
# IMPORT (assumes Part 1 was run)
# =============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix
)
import xgboost as xgb

print("=" * 60)
print("PART 2: MODEL TRAINING")
print("=" * 60)

# =============================================================================
# STEP 1: Train Random Forest
# =============================================================================
print("\n[1/4] Training Random Forest...")
print("      (This may take 30-60 seconds)")

# Random Forest uses BAGGING:
# - Creates 100 independent trees
# - Each tree votes
# - Majority wins

rf = RandomForestClassifier(
    n_estimators=100,    # 100 trees
    random_state=42,     # For reproducibility
    n_jobs=-1            # Use all CPU cores
)

rf.fit(X_train, y_train)
print("✓ Random Forest trained!")

# =============================================================================
# STEP 2: Train XGBoost
# =============================================================================
print("\n[2/4] Training XGBoost...")
print("      (This may take 30-60 seconds)")

# XGBoost uses BOOSTING:
# - Creates trees sequentially
# - Each tree corrects errors of previous tree
# - Weighted sum for final prediction

xgb_model = xgb.XGBClassifier(
    n_estimators=100,    # 100 trees
    max_depth=6,         # Tree depth
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
print("✓ XGBoost trained!")

# =============================================================================
# STEP 3: Make Predictions
# =============================================================================
print("\n[3/4] Making predictions on test set...")

# Random Forest predictions
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]  # Probability of fraud

# XGBoost predictions
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("✓ Predictions complete!")

# =============================================================================
# STEP 4: Evaluate Models
# =============================================================================
print("\n[4/4] Evaluating models...")

print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
print(f"{'Metric':<15} {'Random Forest':>15} {'XGBoost':>15}")
print("-" * 60)

# Calculate metrics for both models
for name, pred, prob in [('Random Forest', rf_pred, rf_prob), 
                          ('XGBoost', xgb_pred, xgb_prob)]:
    auc = roc_auc_score(y_test, prob)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    
    if name == 'Random Forest':
        rf_metrics = {'auc': auc, 'precision': prec, 'recall': rec, 'f1': f1}
    else:
        xgb_metrics = {'auc': auc, 'precision': prec, 'recall': rec, 'f1': f1}

print(f"{'AUC-ROC':<15} {rf_metrics['auc']:>15.4f} {xgb_metrics['auc']:>15.4f}")
print(f"{'Precision':<15} {rf_metrics['precision']:>15.4f} {xgb_metrics['precision']:>15.4f}")
print(f"{'Recall':<15} {rf_metrics['recall']:>15.4f} {xgb_metrics['recall']:>15.4f}")
print(f"{'F1-Score':<15} {rf_metrics['f1']:>15.4f} {xgb_metrics['f1']:>15.4f}")

# Confusion Matrix for Random Forest
print("\n" + "=" * 60)
print("RANDOM FOREST CONFUSION MATRIX")
print("=" * 60)
cm = confusion_matrix(y_test, rf_pred)
print(f"True Negatives:  {cm[0,0]:,} (correctly identified legitimate)")
print(f"False Positives: {cm[0,1]} (legitimate flagged as fraud)")
print(f"False Negatives: {cm[1,0]} (fraud missed)")
print(f"True Positives:  {cm[1,1]} (correctly identified fraud)")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 60)
print("PART 2 COMPLETE")
print("=" * 60)
print("\nKey findings:")
print(f"  - Random Forest AUC: {rf_metrics['auc']:.4f} (BEST)")
print(f"  - Only {cm[0,1]} false positives out of {len(y_test):,} transactions")
print("\nVariables ready for Part 3:")
print("  - rf, xgb_model (trained models)")
print("  - rf_prob, xgb_prob (probabilities)")
print("\nNext: Run Part 3 for the Traffic Light Framework.")


## Next Steps

- Feature engineering refinements
- Robustness evaluation under simulated attacks
- SHAP explainability audit
